# Continuous Joint Distributions

Age–height tables and dice grids were **discrete** joint distributions. What if both variables are *continuous* — say, the **waiting time** before a call-center call is answered and the customer's **satisfaction rating**, each any real number between 0 and 10? The joint distribution becomes a **density surface** $f_{XY}(x, y)$: a mountain landscape over the plane, where probabilities are *volumes* under the surface, and the peaks show where customers actually live — in this story, two corners: "answered fast, very happy" and "waited forever, very unhappy". It is based on the third video of Lesson 4 (Probability Distributions with Multiple Variables).

## 1. Introduction

Recall the phone-call example from Lesson 2: the waiting time for one call was a continuous variable, its distribution a PDF, and probabilities were areas under the curve. Now we track **two** continuous variables per customer:

- $X$: the waiting time before the call is picked up — any value in $[0, 10]$ minutes (after 10 minutes the call ends, so the range is hard);
- $Y$: the customer's satisfaction rating — also continuous in $[0, 10]$ (a rating can be 5.7 or 1.24, not just whole numbers).

After completing this notebook you will be able to:

- Extend the joint-distribution idea from tables to a **joint PDF** $f_{XY}(x, y)$, where region probabilities are volumes.
- Read scatter plots, 2D heatmaps, and the density "mountain" of two continuous variables.
- Compute each variable's **mean** and **variance** from joint data, using the shortcut $\mathrm{Var} = E[Z^2] - E[Z]^2$.

## 2. Theory

### From tables to densities

For discrete pairs, the joint PMF assigned a probability to each *cell*. For continuous pairs, single points have probability zero (just as in one dimension), so the object becomes a **joint probability density function** $f_{XY}(x, y)$ with the two defining properties:

$$
f_{XY}(x, y) \ge 0
\qquad\text{and}\qquad
\iint f_{XY}(x, y)\,dx\,dy = 1
$$

and probabilities are **volumes under the density surface**:

$$
P\big((X, Y) \in \text{region}\big) = \iint_{\text{region}} f_{XY}(x, y)\,dx\,dy
$$

— the 2D analogue of "area under the curve". (As before, you won't need to solve integrals by hand; numerics and intuition carry the weight.)

### The call-center dataset

Each variable alone looks familiar: a histogram of 1,000 waiting times, and a histogram of ratings — with big spikes at 0 and at 10 (many delighted customers, many furious ones).

Looking at **both at once** — each customer a point $(x, y)$ in the square $[0,10]^2$ — reveals what the single histograms cannot: the data concentrates **in two corners**.

- **Low waiting time, high rating**: calls answered quickly → satisfied customers.
- **High waiting time (≈10, or never picked up), low rating**: long waits → unsatisfied customers.

The 2D heatmap shows these two hot regions, and the density can be drawn as a **3D mountain**: imagine the heatmap seen from above, dark areas being high peaks. The peaks are where a random customer is most likely to be found.

### Means and variances from joint data

Aggregate over one variable at a time (over every row for the time distribution, over every column for the ratings), then apply the one-variable machinery from Lesson 3. For the video's dataset:

$$
E[X] = 4.903 \text{ min} \qquad E[Y] = 5.280
$$

The point $(4.903,\ 5.280)$ is the **balance point** — where you would balance the whole scatter of customers on a fingertip. For the spreads, the shortcut formula:

$$
\mathrm{Var}(X) = E[X^2] - E[X]^2 = 32.561 - 4.903^2 = 8.526
$$
$$
\mathrm{Var}(Y) = E[Y^2] - E[Y]^2 = 38.037 - 5.280^2 = 10.163
$$

Note what the means and variances *cannot* see: nothing in these four numbers reveals the two-corner structure or the fact that long waits go with low ratings. Capturing that relationship needs a new tool — the next videos' topic.

## 3. Worked Examples

### Example 1: a uniform joint density

If $(X, Y)$ were uniform on the square $[0,10]^2$, the density would be flat: $f_{XY}(x,y) = \tfrac{1}{100}$ (so the total volume is $100 \cdot \tfrac1{100} = 1$). Then region probabilities are just areas times the density:

$$
P(X \le 2 \text{ and } Y \ge 8) = \underbrace{2 \cdot 2}_{\text{area}} \cdot \tfrac{1}{100} = 0.04
$$

### Example 2: the video's variances, step by step

For $X$: $E[X] = 4.903$ and $E[X^2] = 32.561$, so $\mathrm{Var}(X) = 32.561 - 24.039 = 8.522 \approx 8.526$ (rounding). For $Y$: $38.037 - 27.878 = 10.159 \approx 10.163$. Standard deviations: $\sigma_X \approx 2.92$ min, $\sigma_Y \approx 3.19$ rating points — both large, reflecting the two far-apart clusters.

### Example 3: why the corners matter

Under *independence* the joint density would be (marginal of $X$) × (marginal of $Y$) — spreading mass over **all four** corners (fast-and-angry and slow-and-happy included). The observed density has essentially only two corners: the variables are strongly (negatively) related. Same marginals, completely different joint — the joint holds information no pair of single-variable summaries contains.

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project). We synthesize a dataset with the video's two-corner structure: half the customers get quick calls and give high ratings, half wait long and rate low.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

def make_customers(n, rng):
    quick = rng.random(n) < 0.5                       # half the calls answered fast
    wait = np.where(quick,
                    rng.normal(2.4, 1.6, n),           # quick calls
                    rng.normal(7.6, 1.6, n))           # long waits
    rating = np.where(quick,
                      rng.normal(8.0, 1.7, n),         # happy customers
                      rng.normal(2.6, 1.7, n))         # unhappy customers
    return np.clip(wait, 0, 10), np.clip(rating, 0, 10)

waiting, rating = make_customers(1000, rng)
print(f"first five customers (waiting min, rating): "
      f"{list(zip(waiting[:5].round(1), rating[:5].round(1)))}")

### Means: the balance point of the scatter

In [ ]:
mean_x = waiting.mean()
mean_y = rating.mean()
print(f"E[X] (mean waiting time)   = {mean_x:.3f} minutes   (video: 4.903)")
print(f"E[Y] (mean rating)         = {mean_y:.3f}           (video: 5.280)")
print(f"balance point of the whole scatter: ({mean_x:.3f}, {mean_y:.3f})")

### Variances via the shortcut formula, exactly as in the video

In [ ]:
e_x2 = np.mean(waiting**2)
e_y2 = np.mean(rating**2)
var_x = e_x2 - mean_x**2
var_y = e_y2 - mean_y**2

print(f"E[X^2] = {e_x2:.3f}   ->  Var(X) = {e_x2:.3f} - {mean_x:.3f}^2 = {var_x:.3f}"
      f"   (video: 8.526)")
print(f"E[Y^2] = {e_y2:.3f}   ->  Var(Y) = {e_y2:.3f} - {mean_y:.3f}^2 = {var_y:.3f}"
      f"   (video: 10.163)")
print(f"standard deviations: sigma_X = {np.sqrt(var_x):.2f} min, "
      f"sigma_Y = {np.sqrt(var_y):.2f} points")

### Region probabilities from data: counting customers in a corner

For continuous joints, $P(\text{region})$ is a volume under the density — estimated from data by the *fraction of points* in the region.

In [ ]:
rng = np.random.default_rng(1)
big_wait, big_rate = make_customers(100_000, rng)

quick_happy = np.mean((big_wait <= 3) & (big_rate >= 7))
slow_unhappy = np.mean((big_wait >= 7) & (big_rate <= 3))
quick_unhappy = np.mean((big_wait <= 3) & (big_rate <= 3))

print(f"P(wait <= 3 and rating >= 7)  ~ {quick_happy:.3f}   (quick & happy corner)")
print(f"P(wait >= 7 and rating <= 3)  ~ {slow_unhappy:.3f}   (slow & unhappy corner)")
print(f"P(wait <= 3 and rating <= 3)  ~ {quick_unhappy:.3f}   (quick & unhappy: almost nobody)")

### The uniform-square example, numerically

For the flat density $f = \tfrac{1}{100}$ on $[0,10]^2$, verify the total volume and the corner probability by double numerical integration.

In [ ]:
x = np.linspace(0, 10, 501)
y = np.linspace(0, 10, 501)
density = np.full((len(y), len(x)), 1 / 100)

total = np.trapezoid(np.trapezoid(density, x, axis=1), y)
corner_mask = (x <= 2)[None, :] & (y >= 8)[:, None]
corner = np.trapezoid(np.trapezoid(np.where(corner_mask, density, 0), x, axis=1), y)

print(f"total volume under the uniform density: {total:.4f}")
print(f"P(X <= 2, Y >= 8) = {corner:.4f}   (theory: 4/100 = 0.04)")

## 5. Visualizations

### Each variable alone: the two histograms

The video's starting plots — including the rating's spikes at 0 and 10.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(waiting, bins=30, color="tab:blue", edgecolor="white")
axes[0].set_title("waiting time of 1,000 customers")
axes[0].set_xlabel("waiting time (minutes)")
axes[1].hist(rating, bins=30, color="tab:orange", edgecolor="white")
axes[1].set_title("satisfaction rating of the same customers")
axes[1].set_xlabel("rating")
for ax in axes:
    ax.set_ylabel("count")
    ax.grid(True, alpha=0.3, axis="y")
plt.suptitle("One variable at a time: no hint of the joint structure yet")
plt.tight_layout()
plt.show()

### Both at once: scatter with the balance point

In [ ]:
plt.figure(figsize=(7, 6))
plt.scatter(waiting, rating, s=8, alpha=0.4, color="tab:blue")
plt.plot(mean_x, mean_y, "o", color="tab:red", markersize=12,
         label=f"mean ({mean_x:.2f}, {mean_y:.2f})")
plt.title("1,000 customers: two corners, and the balance point between them")
plt.xlabel("waiting time (minutes)")
plt.ylabel("satisfaction rating")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

### The heatmap and the density mountain

The 2D histogram of 100,000 customers is the empirical joint density; drawn in 3D it becomes the mountain the video describes — two peaks where the two kinds of customers live.

In [ ]:
hist2d, xedges, yedges = np.histogram2d(big_wait, big_rate, bins=40,
                                        range=[[0, 10], [0, 10]], density=True)

fig = plt.figure(figsize=(13, 5))

ax0 = fig.add_subplot(1, 2, 1)
im = ax0.imshow(hist2d.T, origin="lower", extent=[0, 10, 0, 10],
                cmap="viridis", aspect="auto")
ax0.set_title("empirical joint density (heatmap)")
ax0.set_xlabel("waiting time (minutes)")
ax0.set_ylabel("rating")
fig.colorbar(im, ax=ax0, label="density")

ax1 = fig.add_subplot(1, 2, 2, projection="3d")
xc = (xedges[:-1] + xedges[1:]) / 2
yc = (yedges[:-1] + yedges[1:]) / 2
Xg, Yg = np.meshgrid(xc, yc)
ax1.plot_surface(Xg, Yg, hist2d.T, cmap="viridis")
ax1.set_title("the same density as a mountain")
ax1.set_xlabel("waiting time")
ax1.set_ylabel("rating")
ax1.set_zlabel("density")

plt.tight_layout()
plt.show()

### Same marginals, different joint: the independence counterfactual

Shuffle the ratings among customers: each variable's histogram is untouched, but the joint structure dissolves — mass spreads over all four corners. The joint carries information the marginals cannot.

In [ ]:
rng = np.random.default_rng(3)
shuffled_rating = rng.permutation(big_rate)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, yy, title in [(axes[0], big_rate, "real data: two corners (dependent)"),
                      (axes[1], shuffled_rating, "ratings shuffled: four corners (independent)")]:
    h, _, _ = np.histogram2d(big_wait, yy, bins=40, range=[[0, 10], [0, 10]],
                             density=True)
    ax.imshow(h.T, origin="lower", extent=[0, 10, 0, 10], cmap="viridis",
              aspect="auto")
    ax.set_title(title)
    ax.set_xlabel("waiting time (minutes)")
axes[0].set_ylabel("rating")
plt.suptitle("Both panels have IDENTICAL histograms for each variable alone")
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** For the uniform density $f_{XY} = \tfrac{1}{100}$ on $[0,10]^2$, compute (no integrals needed — area × density): $P(X \le 5, Y \le 5)$, $P(X \ge 9)$, and $P(2 \le X \le 4,\ 3 \le Y \le 7)$.

**Exercise 2.** From the 100,000-customer sample, estimate the probability that a customer waits at most 5 minutes *and* gives a rating of at least 5. Is it close to $P(X\le5) \cdot P(Y\ge5)$?

### Intermediate

**Exercise 3.** Define $f_{XY}(x,y) = c \cdot x y$ on the unit square $[0,1]^2$ (zero elsewhere). Find the constant $c$ that makes it a valid density (numerically or by hand), then compute $P(X \le \tfrac12, Y \le \tfrac12)$. Is it $\tfrac14$? Why not?

**Exercise 4.** Compute the video's four summary numbers ($E[X], E[Y], \mathrm{Var}(X), \mathrm{Var}(Y)$) for the *shuffled* dataset of the last visualization. Explain why they are identical to the unshuffled ones — and what that says about the limits of per-variable summaries.

### Advanced

**Exercise 5.** Build the empirical joint density on a grid (`np.histogram2d` with `density=True`) and verify that its total volume is exactly 1 by summing (cell density × cell area) over all cells. Then sum only the cells in the region $x \le 3, y \ge 7$ and compare with the direct point-count estimate from Exercise 2's method.

**Exercise 6.** A continuous joint with *positive* dependence: generate $(X, Y)$ where $Y = 0.8X + \text{noise}$, both clipped to $[0,10]$. Plot its heatmap next to the call-center one. How do the mountain ridges differ, and what sign of relationship does each suggest? (This intuition becomes *covariance* in the next video.)

## 7. Solutions

In [ ]:
# Exercise 1: uniform rectangle probabilities
print(f"Exercise 1: P(X<=5, Y<=5) = 25/100 = {25/100}")
print(f"            P(X>=9)       = (1*10)/100 = {10/100}")
print(f"            P(2<=X<=4, 3<=Y<=7) = (2*4)/100 = {8/100}")

In [ ]:
# Exercise 2: a half-plane corner vs the independence product
p_joint = np.mean((big_wait <= 5) & (big_rate >= 5))
p_product = np.mean(big_wait <= 5) * np.mean(big_rate >= 5)
print(f"Exercise 2: P(X<=5 and Y>=5) ~ {p_joint:.3f}")
print(f"            P(X<=5) * P(Y>=5) = {p_product:.3f}")
print("The joint probability is much larger: quick calls and high ratings travel")
print("together, so the event is far more common than independence would predict.")

In [ ]:
# Exercise 3: normalizing f = c*x*y on the unit square
x = np.linspace(0, 1, 1001)
y = np.linspace(0, 1, 1001)
Xg, Yg = np.meshgrid(x, y)
unnormalized = Xg * Yg
volume = np.trapezoid(np.trapezoid(unnormalized, x, axis=1), y)
c = 1 / volume
print(f"Exercise 3: integral of x*y over the square = {volume:.4f} -> c = {c:.1f}")

quadrant = np.where((Xg <= 0.5) & (Yg <= 0.5), c * unnormalized, 0)
p = np.trapezoid(np.trapezoid(quadrant, x, axis=1), y)
print(f"P(X<=1/2, Y<=1/2) = {p:.4f}  (exact: (1/2)^2 * (1/2)^2 = 1/16)")
print("Not 1/4: the density RISES toward (1,1), so the low corner holds less")
print("volume than its share of area -- unlike the flat uniform density.")

**Exercise 3, illustrated.** A one-page visual summary of the solution above.

<img src="img/03_ex3_normalizing.png" alt="Illustrated solution to Exercise 3" width="900">

In [ ]:
# Exercise 4: per-variable summaries ignore the pairing
for name, yy in [("original", big_rate), ("shuffled", rng.permutation(big_rate))]:
    print(f"{name:>9}: E[X] = {big_wait.mean():.3f}, E[Y] = {yy.mean():.3f}, "
          f"Var(X) = {big_wait.var():.3f}, Var(Y) = {yy.var():.3f}")
print("Identical (the shuffle only reorders Y values, changing no histogram).")
print("Means and variances are computed per variable -- they cannot distinguish")
print("two-corner data from four-corner data. That takes a JOINT summary.")

**Exercise 4, illustrated.** A one-page visual summary of the solution above.

<img src="img/03_ex4_marginals_vs_joint_distribution.png" alt="Illustrated solution to Exercise 4" width="900">

In [ ]:
# Exercise 5: volumes from the empirical density grid
hist2d, xe, ye = np.histogram2d(big_wait, big_rate, bins=60,
                                range=[[0, 10], [0, 10]], density=True)
cell_area = (xe[1] - xe[0]) * (ye[1] - ye[0])
xc = (xe[:-1] + xe[1:]) / 2
yc = (ye[:-1] + ye[1:]) / 2

total = (hist2d * cell_area).sum()
in_region = (xc <= 3)[:, None] & (yc >= 7)[None, :]
p_grid = (hist2d[in_region] * cell_area).sum()
p_count = np.mean((big_wait <= 3) & (big_rate >= 7))

print(f"Exercise 5: total volume = sum(density * cell area) = {total:.4f}")
print(f"corner volume = {p_grid:.4f} vs direct count = {p_count:.4f}")
print("Each histogram cell's (density x area) is exactly its fraction of points,")
print("so the volumes match the counts -- the discrete Riemann sum of the")
print("continuous 'probability = volume under the density' idea.")

**Exercise 5, illustrated.** A one-page visual summary of the solution above.

<img src="img/03_ex5_joint_density.png" alt="Illustrated solution to Exercise 5" width="900">

In [ ]:
# Exercise 6: positive vs negative dependence
rng = np.random.default_rng(5)
x_pos = np.clip(rng.normal(5, 2.6, 100_000), 0, 10)
y_pos = np.clip(0.8 * x_pos + rng.normal(1, 1.4, 100_000), 0, 10)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, xx, yy, title in [
        (axes[0], big_wait, big_rate, "call center: ridge falls -> negative relationship"),
        (axes[1], x_pos, y_pos, "Y = 0.8X + noise: ridge rises -> positive relationship")]:
    h, _, _ = np.histogram2d(xx, yy, bins=40, range=[[0, 10], [0, 10]], density=True)
    ax.imshow(h.T, origin="lower", extent=[0, 10, 0, 10], cmap="viridis", aspect="auto")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("X")
axes[0].set_ylabel("Y")
plt.tight_layout()
plt.show()

print("Exercise 6: the call-center mountain runs from top-left to bottom-right")
print("(large X with small Y); the constructed one runs bottom-left to top-right.")
print("The SIGN of that ridge direction is what covariance will quantify next.")

**Exercise 6, illustrated.** A one-page visual summary of the solution above.

<img src="img/03_ex6_joint_distribution_neg_pos.png" alt="Illustrated solution to Exercise 6" width="900">

## 8. Conclusion

- Continuous pairs are described by a **joint PDF** $f_{XY}(x, y) \ge 0$ with total volume 1; probabilities of regions are **volumes under the density surface** — the 2D version of area under a curve.
- The call-center data ($X$ = waiting time, $Y$ = rating, both continuous in $[0,10]$) concentrates in **two corners**: quick-and-happy, slow-and-unhappy — visible in the scatter, the heatmap, and the 3D density mountain, but in *neither* single histogram.
- Per-variable summaries come from aggregating the joint: $E[X] = 4.903$, $E[Y] = 5.280$ (the balance point of the scatter), and by the shortcut formula $\mathrm{Var}(X) = 32.561 - 4.903^2 = 8.526$, $\mathrm{Var}(Y) = 38.037 - 5.280^2 = 10.163$.
- Means and variances are blind to the relationship: shuffling the ratings leaves all four numbers unchanged while destroying the two-corner structure. Quantifying the relationship itself — the direction of the mountain ridge — is the job of **covariance**, up next.